# 6.7 - Voice AI: STT & TTS

**Module 6 - Image & Multimodal Generation** | Netsetos GenAI Engineering

This notebook works through Voice AI: STT & TTS hands-on: Get text from speech: Whisper locally, Gemini natively; Turn text back into a voice with Kokoro; Wire the loop into one cascaded voice agent; Choose the stack by the constraint that binds.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

The environment prep for this lesson. The heavy voice libraries (`openai-whisper`, `google-genai`, `kokoro`, `soundfile`) are installed per-cell where they are first used, so this top cell only leaves a commented `pip install openai` and a note about seeding - nothing runs.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install openai -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A placeholder setup cell: every line is a comment, so it performs no work. The real installs live inline in the cells that need them (Whisper in cell 2, Kokoro in cell 3), which keeps each concept self-contained.

**How the code works - step by step**
- **Commented `pip install openai`** - uncomment only if you run in Colab or a fresh environment.
- **Reproducibility note** - a reminder that the lesson uses fixed values so outputs are stable; no seed is actually set here.

**In one line:** nothing executes - the installs are deferred to the cells that use each library.

**What you'll see:** (no output - environment prep)

## 1 - Get text from speech: Whisper locally, Gemini natively

Two different routes from audio to meaning. First **Whisper** transcribes a whole file locally and hands back text plus a detected language and timestamped segments. Then **Gemini** takes the audio itself as input and both transcribes *and* reasons over it in a single call - the native-multimodal idea from 6.5, now for sound.

> **Needs a Google API key** (set `GOOGLE_API_KEY`) for the Gemini half; the Whisper half runs locally.

In [ ]:
# pip install openai-whisper google-genai
# --- 1) Whisper: open, local, batch (great for files) ---
import whisper

model = whisper.load_model("large-v3-turbo")   # turbo: ~5x faster decoder, near-v3 accuracy
result = model.transcribe("meeting.mp3")        # transcribes the WHOLE clip, then returns
print(result["text"][:54], "...")
print("detected language:", result["language"])
for seg in result["segments"][:1]:      # timestamps, e.g. for subtitles
    print(f"[{seg['start']:.1f}s-{seg['end']:.1f}s] {seg['text'].strip()}")
# Output: Welcome everyone, let's get started with the quarterly ...
# Output: detected language: en
# Output: [0.0s-3.4s] Welcome everyone, let's get started with the quarterly review.

# --- 2) Gemini: send audio straight to an LLM (transcribe AND understand) ---
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
with open("question.wav", "rb") as f:
    audio = f.read()

resp = client.models.generate_content(
    model="gemini-3-flash",
    contents=["Transcribe this, then answer the question it asks.",
              types.Part.from_bytes(data=audio, mime_type="audio/wav")],
)
print(resp.text)
# Output: You asked "what's the capital of Japan?" - it is Tokyo.

**Explanation**

This cell contrasts two philosophies of speech-to-text side by side. The Whisper block is pure transcription - a finished clip in, structured text out - while the Gemini block skips the separate transcript step and sends raw audio bytes straight into an LLM that answers the spoken question. Read it as: one path gives you words, the other gives you understanding.

**How the code works - step by step**
- **`whisper.load_model("large-v3-turbo")`** - loads the open, local model; `turbo` shrinks the decoder for ~5x speed at near-v3 accuracy.
- **`model.transcribe("meeting.mp3")`** - reads the *entire* clip (batch, not streaming), returning a dict with `text`, `language`, and timestamped `segments`.
- **Printing `segments`** - each segment carries `start`/`end` seconds and its text - exactly what you need for subtitles.
- **`genai.Client(...)`** - opens the unified google-genai SDK (the older `google.generativeai` was deprecated).
- **`types.Part.from_bytes(...)`** - wraps the raw `.wav` bytes so they ride into the model alongside a text instruction.
- **`generate_content(model="gemini-3-flash", ...)`** - transcribes and answers the spoken question in one round-trip.

**In one line:** Whisper gives you a transcript; Gemini native audio gives you an answer.

**What you'll see:** The Whisper block prints the first chars of the transcript ("Welcome everyone, let's get started with the quarterly ..."), `detected language: en`, and one timestamped segment `[0.0s-3.4s]`; the Gemini block prints its spoken-question answer, e.g. `You asked "what's the capital of Japan?" - it is Tokyo.`

## 2 - Turn text back into a voice with Kokoro

The other half of the loop: text in, waveform out. **Kokoro** is the open, CPU-friendly TTS default (82M params, Apache-2.0, preset voices only - no cloning). Here it speaks one line and writes it to a 24 kHz `.wav`, standing in for the acoustic-model-plus-vocoder pipeline as a single call.

In [ ]:
# pip install kokoro soundfile   -  open TTS (Apache-2.0), runs on CPU, 82M params
from kokoro import KPipeline
import soundfile as sf

pipeline = KPipeline(lang_code="a")   # 'a' = American English (preset voices, no cloning)
text = "Welcome to the course. Your voice AI journey starts here."

# The pipeline yields (graphemes, phonemes, audio) chunks; write each to a wav.
for i, (graphemes, phonemes, audio) in enumerate(pipeline(text, voice="af_heart")):
    sf.write(f"line_{i}.wav", audio, 24000)   # 24 kHz waveform
print("saved line_0.wav (24 kHz)")
# Output: saved line_0.wav (24 kHz)

**Explanation**

A minimal text-to-speech example that produces a real audio file. `KPipeline` is the whole acoustic-model-plus-vocoder chain wrapped in one iterator: you feed it text and a preset voice, and it yields audio chunks you write to disk. It is deliberately the simplest possible TTS call - no SSML, no cloning - so the mechanics are visible.

**How the code works - step by step**
- **`KPipeline(lang_code="a")`** - builds the pipeline for American English using preset voices (no cloning capability).
- **`pipeline(text, voice="af_heart")`** - runs synthesis, yielding `(graphemes, phonemes, audio)` tuples; the phonemes are the individual speech sounds behind each word.
- **`sf.write(f"line_{i}.wav", audio, 24000)`** - writes each returned chunk as a 24 kHz waveform.

**In one line:** text -> phonemes -> 24 kHz waveform, written to a `.wav`.

**What you'll see:** Prints `saved line_0.wav (24 kHz)` and leaves a playable `line_0.wav` file on disk.

## 3 - Wire the loop into one cascaded voice agent

Now assemble the full pipeline as a class. `VoiceAgent` holds all three pieces - Whisper (ears), Gemini (brain), Kokoro (voice) - and runs one conversational turn: hear the user, think of a reply, speak it back. This is the **cascaded** design: three visible, swappable, loggable stages.

> **Needs a Google API key** (set `GOOGLE_API_KEY`) and the whisper + kokoro libraries.

In [ ]:
# A cascaded voice agent: STT -> LLM -> TTS. Frameworks (LiveKit, Pipecat)
# handle the mic, WebRTC, and turn-taking; here is the core loop, decomposed.
import os, whisper, soundfile as sf
import numpy as np
from google import genai
from kokoro import KPipeline

class VoiceAgent:
    def __init__(self):
        self.stt = whisper.load_model("large-v3-turbo")          # 1. ears
        self.llm = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])  # 2. brain
        self.tts = KPipeline(lang_code="a")                         # 3. voice

    def transcribe(self, wav_path: str) -> str:
        return self.stt.transcribe(wav_path)["text"].strip()   # you can LOG this

    def think(self, user_text: str) -> str:
        reply = self.llm.models.generate_content(
            model="gemini-3-flash",
            contents=[f"Reply in one short spoken sentence: {user_text}"],
        )
        return reply.text.strip()                            # and LOG this

    def speak(self, text: str, out="reply.wav"):
        # Kokoro yields ONE chunk per sentence - concatenate them all, then
        # write ONE file. A naive sf.write() inside the loop would overwrite
        # the file each pass and keep only the last chunk.
        parts = [audio for _, _, audio in self.tts(text, voice="af_heart")]
        sf.write(out, np.concatenate(parts), 24000)
        return out

    def turn(self, wav_path: str) -> str:
        heard = self.transcribe(wav_path)                    # STT
        reply = self.think(heard)                            # LLM
        print(f"heard: {heard}\nreply: {reply}")
        return self.speak(reply)                             # TTS

agent = VoiceAgent()
agent.turn("user_question.wav")
# Output: heard: what is the capital of France
# Output: reply: The capital of France is Paris.

**Explanation**

This is the lesson's centrepiece: the three separate tools from the previous cells fused into one object with a single `turn` method. The whole point of the cascaded architecture is on display - each stage (`transcribe`, `think`, `speak`) is its own method you can log, redact, or swap independently, unlike a black-box speech-to-speech model. It is a structural example, not a new capability.

**How the code works - step by step**
- **`__init__`** - loads the three models once: Whisper STT, the Gemini client, and the Kokoro pipeline.
- **`transcribe`** - runs STT on a wav path and returns the stripped text (a stage you can log).
- **`think`** - sends the heard text to `gemini-3-flash` with a 'reply in one short spoken sentence' instruction (also loggable).
- **`speak`** - collects *all* Kokoro chunks, `np.concatenate`s them, and writes ONE file - the inline comment warns that a naive `sf.write` inside the loop would overwrite and keep only the last chunk.
- **`turn`** - the orchestrator: transcribe -> think -> print both -> speak, returning the reply wav path.

**In one line:** hear (STT) -> think (LLM) -> speak (TTS), each stage visible and swappable.

**What you'll see:** Prints the two intermediate stages - `heard: what is the capital of France` and `reply: The capital of France is Paris.` - and writes `reply.wav`.

## 4 - Choose the stack by the constraint that binds

A decision helper, not a model call. Given a need - private files, real-time STT, understanding, open TTS, cloning - it returns the tool to reach for as of mid-2026. The lesson's point is baked into its shape: you pick voice tools by constraint (batch vs real-time, private vs hosted, cloning or not), because WER and MOS are close at the top.

In [ ]:
# Pick voice tools by the constraint that binds - not by one benchmark.
def choose_voice_stack(need: str) -> str:
    table = {
        "files_private":  "Whisper large-v3-turbo (open, batch, timestamps)",
        "realtime_stt":   "Deepgram Nova-3 / Flux (streaming, end-of-turn)",
        "understand":     "Gemini native audio (transcribe + reason in one call)",
        "tts_open":       "Kokoro (82M, Apache-2.0, runs on CPU)",
        "tts_cloning":    "F5-TTS (open) or ElevenLabs (hosted) - mind the license",
    }
    return table.get(need, "start open + cascaded; add hosted/streaming when latency or scale forces it")

for need in ["files_private", "realtime_stt", "tts_open", "tts_cloning"]:
    print(f"{need:14} -> {choose_voice_stack(need)}")
# Output: files_private  -> Whisper large-v3-turbo (open, batch, timestamps)
# Output: realtime_stt   -> Deepgram Nova-3 / Flux (streaming, end-of-turn)
# Output: tts_open       -> Kokoro (82M, Apache-2.0, runs on CPU)
# Output: tts_cloning    -> F5-TTS (open) or ElevenLabs (hosted) - mind the license

**Explanation**

A pure lookup function that encodes the lesson's selection logic - no audio, no network, just a dictionary from constraint to recommendation. It exists to make the 'constraint, not benchmark' rule executable and memorable, with a sensible fallback for unrecognised needs.

**How the code works - step by step**
- **`table` dict** - maps each constraint key (`files_private`, `realtime_stt`, `understand`, `tts_open`, `tts_cloning`) to a concrete tool and one-line rationale.
- **`table.get(need, ...)`** - returns the match, or a default 'start open + cascaded; add hosted/streaming when latency or scale forces it'.
- **The `for` loop** - prints four representative needs against their recommended tools.

**In one line:** name your binding constraint -> get the tool, not a benchmark winner.

**What you'll see:** Prints four aligned rows mapping each need to its pick, e.g. `files_private  -> Whisper large-v3-turbo (open, batch, timestamps)` through `tts_cloning    -> F5-TTS (open) or ElevenLabs (hosted) - mind the license`.

These five cells walk the full voice loop end to end: Whisper and Gemini turn speech into text (and understanding), Kokoro turns text back into a waveform, the `VoiceAgent` class wires all three into one auditable turn, and `choose_voice_stack` reduces the whole 2026 landscape to a constraint-driven lookup. The throughline is that a voice AI is never one model - it is STT plus an LLM plus TTS, and every design choice (batch vs streaming, cascaded vs speech-to-speech, open vs hosted) is a latency-and-visibility tradeoff. Module 11 scales this loop into production agents, and Module 15 develops the consent, disclosure, and watermarking that a biometric voice demands.